In [ ]:
#@title 1 · Install (run all -> this runs automatically)
!git clone -q -b dev https://github.com/camenduru/OpenVoice
%cd /content/OpenVoice
!pip install -q -e .
!pip install -q langid faster-whisper whisper-timestamped unidecode eng-to-ipa pypinyin cn2an wavmark
!pip install -q fastapi "uvicorn[standard]" python-multipart requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("deps installed")


In [ ]:
#@title 2 · Download checkpoints
%cd /content/OpenVoice
!wget -q https://huggingface.co/camenduru/OpenVoice/resolve/main/checkpoints_1226.zip -O checkpoints_1226.zip
!unzip -q -o checkpoints_1226.zip
!ls checkpoints/base_speakers/EN
print("checkpoints ready")


In [ ]:
#@title 3 · Write the API server
%%writefile /content/OpenVoice/server.py
import os
import torch
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from openvoice import se_extractor
from openvoice.api import BaseSpeakerTTS, ToneColorConverter

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CKPT_BASE = "checkpoints/base_speakers/EN"
CKPT_CONV = "checkpoints/converter"
EMOTIONS = ["default", "friendly", "cheerful", "excited",
            "sad", "angry", "terrified", "shouting", "whispering"]

print(f"[init] loading OpenVoice on {DEVICE} ...")
base = BaseSpeakerTTS(f"{CKPT_BASE}/config.json", device=DEVICE)
base.load_ckpt(f"{CKPT_BASE}/checkpoint.pth")
converter = ToneColorConverter(f"{CKPT_CONV}/config.json", device=DEVICE)
converter.load_ckpt(f"{CKPT_CONV}/checkpoint.pth")
SRC_DEFAULT = torch.load(f"{CKPT_BASE}/en_default_se.pth").to(DEVICE)
SRC_STYLE = torch.load(f"{CKPT_BASE}/en_style_se.pth").to(DEVICE)
os.makedirs("outputs", exist_ok=True)
os.makedirs("processed", exist_ok=True)
print("[init] ready.")

app = FastAPI(title="OpenVoice One-Shot Clone API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE, "emotions": EMOTIONS}

@app.post("/clone")
async def clone(text: str = Form(...), emotion: str = Form("default"),
                speed: float = Form(1.0), reference: UploadFile = File(...)):
    if emotion not in EMOTIONS:
        raise HTTPException(400, f"emotion must be one of {EMOTIONS}")
    if not text.strip():
        raise HTTPException(400, "text is empty")
    suffix = os.path.splitext(reference.filename or "ref.wav")[1] or ".wav"
    ref_path = f"processed/ref_input{suffix}"
    with open(ref_path, "wb") as f:
        f.write(await reference.read())
    target_se, _ = se_extractor.get_se(ref_path, converter, target_dir="processed", vad=True)
    src_path = "outputs/_src.wav"
    base.tts(text, src_path, speaker=emotion, language="English", speed=float(speed))
    out_path = "outputs/cloned.wav"
    src_se = SRC_DEFAULT if emotion == "default" else SRC_STYLE
    converter.convert(audio_src_path=src_path, src_se=src_se, tgt_se=target_se,
                      output_path=out_path, message="@MyShell")
    return FileResponse(out_path, media_type="audio/wav", filename="cloned.wav")


In [ ]:
#@title 4 · Launch API + print PUBLIC_API_URL
import os, re, time, threading, subprocess, requests, uvicorn
os.chdir("/content/OpenVoice")
def _run(): uvicorn.run("server:app", host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_run, daemon=True).start()
print("loading model", end="")
for _ in range(150):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("  ready!"); break
    except Exception: pass
    print(".", end="", flush=True); time.sleep(2)
proc = subprocess.Popen(["cloudflared","tunnel","--url","http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
URL=None
for line in proc.stdout:
    m=re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m: URL=m.group(0); break
print("\n"+"="*62)
print("  PUBLIC_API_URL=" + str(URL))
print("="*62)
